# STREAM-1D Solver Verification & Visualisation

This notebook demonstrates and verifies the **STREAM-1D** native Python bindings. By compiling the identical Rust core used in the WebAssembly module, the Python module runs high-fidelity hydraulic computations (open channel flow, composite roughness, and structure losses) with zero execution overhead.

Here, we will:
1. Run a steady-state profile verification on the HEC-RAS ConSpan test project (Q = 1000 cfs, featuring an arch culvert).
2. Plot the resulting water surface profile (WSEL) and critical depths.
3. Run an integrated bridge pier loss simulation demonstrating Yarnell head loss calculations.
4. Verify the **Issaquah01** bridge reach against HEC-RAS steady profiles (two bridges, deck vents removed).
5. Tabulate results in Pandas DataFrames and plot WSEL overlays for both cases.


In [ ]:
import sys
import importlib.util
from pathlib import Path


def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "verification" / "fixtures" / "conspan_project_12.json").is_file():
        return cwd
    if cwd.name == "python" and (
        cwd.parent / "verification" / "fixtures" / "conspan_project_12.json"
    ).is_file():
        return cwd.parent
    raise RuntimeError(
        "Could not locate repo root (verification/fixtures/conspan_project_12.json). "
        "Open this notebook from python/ or run Jupyter with cwd set to python/."
    )


REPO_ROOT = _repo_root()
PYTHON_DIR = REPO_ROOT / "python"
FIXTURES = REPO_ROOT / "verification" / "fixtures"

# When Jupyter cwd is python/, that folder shadows site-packages unless we prefer the
# maturin-installed extension (Binder postBuild also copies _stream1d next to source).
if Path.cwd().resolve() == PYTHON_DIR:
    spec = importlib.util.find_spec("stream1d._stream1d")
    if spec is not None and spec.origin and str(PYTHON_DIR) not in spec.origin:
        sys.path = [p for p in sys.path if Path(p).resolve() != PYTHON_DIR]

try:
    import stream1d as st
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "The stream1d Python extension is not installed in this kernel.\n\n"
        "From the repository root (not python/), run:\n"
        "  python3 scripts/run_verification_notebook.py\n"
        "or:\n"
        "  source .venv/bin/activate\n"
        "  pip install -r requirements.txt\n"
        "  maturin develop --features python --release\n\n"
        "Select the .venv kernel in Jupyter, then restart and run all cells."
    ) from exc

import json
import pandas as pd
import matplotlib.pyplot as plt

print("STREAM-1D package imported successfully!")
print(f"Repo root: {REPO_ROOT}")

## 1. Load HEC-RAS ConSpan Verification Project Data

We load the ConSpan verification geometry and extract the culvert parameters.

In [ ]:
dataset_path = FIXTURES / "conspan_project_12.json"
with open(dataset_path, "r") as f:
    project = json.load(f)

# Extract geometries and map to stream1d.CrossSection objects
xs_raw = project["geometry_data"]
cross_sections = []
for xs in xs_raw:
    cross_sections.append(
        st.CrossSection(
            station=float(xs["station"]),
            x=[float(val) for val in xs["x"]],
            y=[float(val) for val in xs["y"]],
            n_stations=[float(val) for val in xs["n_stations"]],
            n_values=[float(val) for val in xs["n_values"]],
            unit_system=xs.get("unit_system", "USCustomary"),
            is_overbank=xs.get("is_overbank")
        )
    )

# Extract culvert data
culverts = project.get("culvert_stations", [])
culvert_stations = [float(c["station"]) for c in culverts]
culvert_shape_types = [int(c["shape_type"]) for c in culverts]
culvert_spans = [float(c["span"]) for c in culverts]
culvert_rises = [float(c["rise"]) for c in culverts]
culvert_roughness_ns = [float(c["roughness_n"]) for c in culverts]
culvert_lengths = [float(c["length"]) for c in culverts]
culvert_entrance_loss_coeffs = [float(c["entrance_loss_coeff"]) for c in culverts]
culvert_exit_loss_coeffs = [float(c["exit_loss_coeff"]) for c in culverts]
culvert_barrels = [int(c.get("num_barrels", 1)) for c in culverts]
culvert_roughness_n_bottoms = [float(c.get("roughness_n_bottom", c["roughness_n"])) for c in culverts]
culvert_depth_bottom_ns = [float(c.get("depth_bottom_n", 0.0)) for c in culverts]
culvert_depth_blockeds = [float(c.get("depth_blocked", 0.0)) for c in culverts]

print(f"Loaded {len(cross_sections)} cross-sections and {len(culvert_stations)} culvert structures.")

## 2. Solve Steady-State Profile (Q = 1000 cfs)

We instantiate [SteadyInputs](file:///) and execute the standard-step backwater calculation.

In [ ]:
inputs = st.SteadyInputs(
    cross_sections=cross_sections,
    flow_rate=1000.0,
    num_slices=int(project["parameters"].get("vertical_slices", 100)),
    coeff_contraction=0.1,
    coeff_expansion=0.3,
    regime=int(project["parameters"].get("flow_regime", 0)),
    downstream_wsel=30.51,
    max_spacing=float(project["parameters"].get("max_spacing", 100.0)),
    # Boundary conditions
    downstream_bc_type=0, # Known WSEL
    downstream_bc_slope=float(project["parameters"].get("downstream_bc_slope", 0.001)),
    downstream_bc_rating_q=[],
    downstream_bc_rating_wsel=[],
    upstream_bc_type=0,
    upstream_bc_slope=float(project["parameters"].get("upstream_bc_slope", 0.01)),
    upstream_bc_rating_q=[],
    upstream_bc_rating_wsel=[],
    upstream_wsel=float(project["parameters"].get("upstream_wsel", 15.0)),
    # Culverts
    culvert_stations=culvert_stations,
    culvert_shape_types=culvert_shape_types,
    culvert_spans=culvert_spans,
    culvert_rises=culvert_rises,
    culvert_roughness_ns=culvert_roughness_ns,
    culvert_lengths=culvert_lengths,
    culvert_entrance_loss_coeffs=culvert_entrance_loss_coeffs,
    culvert_exit_loss_coeffs=culvert_exit_loss_coeffs,
    culvert_barrels=culvert_barrels,
    culvert_roughness_n_bottoms=culvert_roughness_n_bottoms,
    culvert_depth_bottom_ns=culvert_depth_bottom_ns,
    culvert_depth_blockeds=culvert_depth_blockeds,
)

results = st.solve_steady(inputs)
print("Steady sweep completed successfully.")

## 3. Compare with HEC-RAS Expected Results

Let's compare STREAM-1D calculations against HEC-RAS.

In [ ]:
# Verification target elevations
expected_wsel = {
    2827.0: 33.720,
    1257.0: 32.920,
    0.0: 30.510
}

station_list = [float(xs["station"]) for xs in xs_raw]
table_data = []
for station, expected_val in expected_wsel.items():
    if station in station_list:
        idx = station_list.index(station)
        calc_val = results["wsel"][idx]
        diff = calc_val - expected_val
        table_data.append({
            "Station": int(station),
            "Calculated WSEL (ft)": round(calc_val, 3),
            "HEC-RAS WSEL (ft)": round(expected_val, 3),
            "Difference (ft)": round(diff, 3),
            "Status": "PASS" if abs(diff) <= 0.04 else "FAIL"
        })

df = pd.DataFrame(table_data)
print(df.to_string(index=False))

failures = [row for row in table_data if row["Status"] == "FAIL"]
if failures:
    raise AssertionError(
        f"ConSpan WSEL verification failed at {len(failures)} station(s): {failures}"
    )

## 4. Plot Water Surface Profile

We plot the channel bed invert, STREAM-1D and HEC-RAS water surface profiles, and critical depth elevations along the reach.

In [ ]:
import csv

stations = [float(xs.station) for xs in cross_sections]
bed_inverts = [min(xs.y) for xs in cross_sections]
wsel = results["wsel"]
crit_wsel = results["critical_wsel"]

# HEC-RAS reference profile: 50 yr event, Q = 1000 cfs (ConSpan.csv)
hecras_stations = []
hecras_wsel = []
hecras_crit_wsel = []
current_sta = None
with open(FIXTURES / "ConSpan.csv", newline="") as f:
    reader = csv.reader(f)
    next(reader)  # column headers
    next(reader)  # units row
    for row in reader:
        if not row or not any(cell.strip() for cell in row):
            continue
        if row[0].strip():
            current_sta = float(row[0].strip())
        profile = row[1].strip() if len(row) > 1 else ""
        if profile == "50 yr" and current_sta is not None:
            hecras_stations.append(current_sta)
            hecras_wsel.append(float(row[4]))
            hecras_crit_wsel.append(float(row[5]))

plt.figure(figsize=(10, 6))
plt.plot(stations, bed_inverts, label="Bed Invert", color="black", linewidth=2)
plt.plot(stations, wsel, label="STREAM-1D WSEL", color="dodgerblue", linewidth=2.5)
plt.plot(
    hecras_stations,
    hecras_wsel,
    label="HEC-RAS WSEL (50 yr, 1000 cfs)",
    color="darkorange",
    linestyle="--",
    linewidth=2.5,
    marker="o",
    markersize=4,
)
plt.plot(stations, crit_wsel, label="STREAM-1D Critical WSEL", color="red", linestyle="--", linewidth=1.5)
plt.plot(
    hecras_stations,
    hecras_crit_wsel,
    label="HEC-RAS Critical WSEL",
    color="crimson",
    linestyle=":",
    linewidth=1.5,
    marker="s",
    markersize=3,
)

# Highlight culvert zone
plt.axvspan(1200, 1257, color="gray", alpha=0.3, label="Culvert Obstruction Zone")

plt.title("Water Surface Profile Comparison — STREAM-1D vs HEC-RAS (ConSpan)", fontsize=14)
plt.xlabel("Stationing (ft)", fontsize=12)
plt.ylabel("Elevation (ft)", fontsize=12)
plt.legend(loc="upper right", fontsize=9)
plt.grid(True, linestyle=":", alpha=0.6)
plt.gca().invert_xaxis()  # Downstream is 0 (right), upstream is positive (left)
plt.show()

## 5. Verify Bridge & Pier Hydraulics

This simulation validates the **Yarnell Pier head loss equation** on a 3-section reach with a bridge deck and square pier obstruction.

In [ ]:
# Setup a 200m reach with a bridge deck at station 50
xs200 = st.CrossSection(200.0, [0.0, 0.0, 10.0, 10.0], [10.2, 0.2, 0.2, 10.2], [0.0], [0.03], "Metric")
xs100 = st.CrossSection(100.0, [0.0, 0.0, 10.0, 10.0], [10.1, 0.1, 0.1, 10.1], [0.0], [0.03], "Metric")
xs0 = st.CrossSection(0.0, [0.0, 0.0, 10.0, 10.0], [10.0, 0.0, 0.0, 10.0], [0.0], [0.03], "Metric")

bridge_inputs = st.SteadyInputs(
    cross_sections=[xs200, xs100, xs0],
    flow_rate=15.0,  # 15 cms
    num_slices=50,
    regime=0,        # Subcritical
    downstream_wsel=3.0,
    downstream_bc_type=0,
    bridge_stations=[50.0],
    bridge_low_chords=[5.0],
    bridge_high_chords=[7.0],
    bridge_pier_widths=[0.5], # 0.5m pier width
    bridge_num_piers=[2],
    bridge_pier_shapes=[0],
    bridge_weir_coeffs=[1.44],
    bridge_orifice_coeffs=[0.5]
)

bridge_results = st.solve_steady(bridge_inputs)
ds_wsel = bridge_results["wsel"][2]
us_wsel = bridge_results["wsel"][1]
backwater = us_wsel - ds_wsel

print(f"Bridge Downstream WSEL (Sta 0):   {ds_wsel:.4f} m")
print(f"Bridge Upstream WSEL (Sta 100):   {us_wsel:.4f} m")
print(f"Pier Headwater Backwater Loss:    {backwater:.4f} m")

## 6. Issaquah01 Bridge Reach — HEC-RAS Steady Parity

The **Issaquah01** example is a seven-cross-section reach with **two bridges** (river stations 500 and 350). Geometry is exported from the Stream1D workspace (`examples/Issaquah01_stream1d.json`). Deck vents are removed so the case aligns with standard HEC-RAS 1D bridge hydraulics.

Each profile uses:
- **Downstream BC:** normal depth with friction slope **Sf = 0.005**
- **Numerical controls:** 100 vertical slices, 100 ft max spacing (same as the web app defaults)

Reference values are from HEC-RAS steady profile output at computational stations (`examples/Issaquah01_profiles.json`). Verification follows the same workflow as the ConSpan case above: comparison tables, then longitudinal profile plots with WSEL and critical depth.

In [ ]:
import copy

sys.path.insert(0, str(REPO_ROOT / "verification" / "oracle"))
from lib.geometry_fixture_mapper import (  # noqa: E402
    load_geometry_fixture,
    river_station_to_computational,
    run_profile,
)

ISSAQUAH_GEOMETRY = REPO_ROOT / "examples" / "Issaquah01_stream1d.json"
ISSAQUAH_PROFILES = REPO_ROOT / "examples" / "Issaquah01_profiles.json"

with open(ISSAQUAH_PROFILES, "r", encoding="utf-8") as fh:
    issaquah_doc = json.load(fh)

issaquah_fixture = copy.deepcopy(load_geometry_fixture(ISSAQUAH_GEOMETRY))
if issaquah_doc.get("strip_deck_vents"):
    for bridge in issaquah_fixture.get("bridge_data", []):
        bridge["deck_vents"] = []
        bridge.pop("deck_vent_pattern", None)

issaquah_geo = issaquah_fixture["geometry_data"]
issaquah_stations = [float(xs["station"]) for xs in issaquah_geo]
issaquah_bed = [min(float(y) for y in xs["y"]) for xs in issaquah_geo]
issaquah_bridge_stations = sorted(
    (
        river_station_to_computational(float(b["station"]), issaquah_geo)
        for b in issaquah_fixture.get("bridge_data", [])
    ),
    reverse=True,
)
issaquah_tol_wsel = float(issaquah_doc.get("tolerance_wsel_ft", 0.05))
issaquah_tol_egl = float(issaquah_doc.get("tolerance_egl_ft", 0.05))

print(
    f"Loaded Issaquah01: {len(issaquah_geo)} cross-sections, "
    f"{len(issaquah_fixture.get('bridge_data', []))} bridges, "
    f"{len(issaquah_doc['profiles'])} verification profile(s)."
)

### 6.1 Compare with HEC-RAS Expected Results

For each flow, tabulate STREAM-1D vs HEC-RAS at every cross section (same table format as the ConSpan verification above). WSEL and EGL must pass within ±0.05 ft.

In [ ]:
def _hecras_comparison_table(
    run: dict,
    expected: dict,
    calc_key: str,
    value_label: str,
    tol: float,
) -> list[dict]:
    by_sta = {round(row["computational_station"], 1): row for row in run["stations"]}
    rows = []
    for key in sorted(expected.keys(), key=float, reverse=True):
        sta = float(key)
        calc_val = float(by_sta[round(sta, 1)][calc_key])
        expected_val = float(expected[key])
        diff = calc_val - expected_val
        rows.append(
            {
                "Station": sta,
                f"Calculated {value_label} (ft)": round(calc_val, 3),
                f"HEC-RAS {value_label} (ft)": round(expected_val, 3),
                "Difference (ft)": round(diff, 3),
                "Status": "PASS" if abs(diff) <= tol else "FAIL",
            }
        )
    return rows


issaquah_runs = {}
issaquah_failures = []

for profile in issaquah_doc["profiles"]:
    run = run_profile(issaquah_fixture, profile)
    issaquah_runs[profile["name"]] = run
    q_label = f"Q = {profile['flow_rate_cfs']:.0f} cfs"
    print(f"\n=== {profile['name']} — {q_label} — WSEL ===")
    wsel_rows = _hecras_comparison_table(
        run,
        profile["expected_wsel_ft"],
        "wsel_ft",
        "WSEL",
        issaquah_tol_wsel,
    )
    print(pd.DataFrame(wsel_rows).to_string(index=False))

    print(f"\n=== {profile['name']} — {q_label} — EGL ===")
    egl_rows = _hecras_comparison_table(
        run,
        profile["expected_egl_ft"],
        "egl_ft",
        "EGL",
        issaquah_tol_egl,
    )
    print(pd.DataFrame(egl_rows).to_string(index=False))

    for row in wsel_rows + egl_rows:
        if row["Status"] == "FAIL":
            issaquah_failures.append((profile["name"], row))

if issaquah_failures:
    raise AssertionError(
        f"Issaquah01 verification failed at {len(issaquah_failures)} station(s): {issaquah_failures}"
    )

print("\nIssaquah01 WSEL/EGL checks passed within tolerance.")

### 6.2 Plot Water Surface Profiles

Plot the channel bed invert, STREAM-1D and HEC-RAS water surface profiles, and critical depth elevations along the reach (one figure per flow, matching the ConSpan plot style).

In [ ]:
bridge_span = (
    min(issaquah_bridge_stations) - 20.0,
    max(issaquah_bridge_stations) + 20.0,
)

for profile in issaquah_doc["profiles"]:
    run = issaquah_runs[profile["name"]]
    stream_wsel = [row["wsel_ft"] for row in run["stations"]]
    stream_crit_wsel = [row["critical_wsel_ft"] for row in run["stations"]]

    hec_keys = sorted(profile["expected_wsel_ft"].keys(), key=float, reverse=True)
    hec_stations = [float(k) for k in hec_keys]
    hec_wsel = [float(profile["expected_wsel_ft"][k]) for k in hec_keys]
    hec_crit_wsel = [float(profile["expected_crit_wsel_ft"][k]) for k in hec_keys]

    q = profile["flow_rate_cfs"]
    plt.figure(figsize=(10, 6))
    plt.plot(issaquah_stations, issaquah_bed, label="Bed Invert", color="black", linewidth=2)
    plt.plot(
        issaquah_stations,
        stream_wsel,
        label="STREAM-1D WSEL",
        color="dodgerblue",
        linewidth=2.5,
    )
    plt.plot(
        hec_stations,
        hec_wsel,
        label=f"HEC-RAS WSEL (Q = {q:.0f} cfs)",
        color="darkorange",
        linestyle="--",
        linewidth=2.5,
        marker="o",
        markersize=4,
    )
    plt.plot(
        issaquah_stations,
        stream_crit_wsel,
        label="STREAM-1D Critical WSEL",
        color="red",
        linestyle="--",
        linewidth=1.5,
    )
    plt.plot(
        hec_stations,
        hec_crit_wsel,
        label="HEC-RAS Critical WSEL",
        color="crimson",
        linestyle=":",
        linewidth=1.5,
        marker="s",
        markersize=3,
    )
    plt.axvspan(
        bridge_span[0],
        bridge_span[1],
        color="gray",
        alpha=0.3,
        label="Bridge Obstruction Zone",
    )
    plt.title(
        f"Water Surface Profile Comparison — STREAM-1D vs HEC-RAS (Issaquah01, Q = {q:.0f} cfs)",
        fontsize=14,
    )
    plt.xlabel("Distance (ft)", fontsize=12)
    plt.ylabel("Elevation (ft)", fontsize=12)
    plt.legend(loc="upper right", fontsize=9)
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.gca().invert_xaxis()  # Downstream is on the right
    plt.show()